In [ ]:
import os
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import json
from scipy.stats import norm

# --- PREPROCESAMIENTO ---
def mejor_roi_por_negros(image, ventana=500, paso=20):
    H, W = image.shape
    w = min(ventana, W, H)
    h = min(ventana, W, H)
    mejor_ventana = (0, 0, w, h)
    mejor_suma = -1
    for y in range(0, H - h + 1, paso):
        for x in range(0, W - w + 1, paso):
            region = image[y:y+h, x:x+w]
            suma = np.sum(region == 0)
            if suma > mejor_suma:
                mejor_suma = suma
                mejor_ventana = (x, y, w, h)
    return mejor_ventana

def recortar_roi(db_path, out_path, ventana=500, paso=15):
    if not os.path.exists(db_path): return
    items = os.listdir(db_path)
    images = [f for f in items if f.lower().endswith(('.png', '.jpg'))]
    users = [d for d in items if os.path.isdir(os.path.join(db_path, d))]
    tareas = []
    if images:
        dst_folder = os.path.join(out_path, "roi")
        tareas.append((db_path, dst_folder))
    else:
        for user in users:
            tareas.append((os.path.join(db_path, user), os.path.join(out_path, user, "roi")))
    for src_dir, dst_dir in tareas:
        os.makedirs(dst_dir, exist_ok=True)
        for filename in [f for f in os.listdir(src_dir) if f.lower().endswith(".png")]:
            img = cv.imread(os.path.join(src_dir, filename))
            if img is None: continue
            gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
            _, image = cv.threshold(gray, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
            x, y, w, h = mejor_roi_por_negros(image, ventana=ventana, paso=paso)
            cv.imwrite(os.path.join(dst_dir, filename), img[y:y+h, x:x+w])

def procesar_fase(fase_func, input_folder, output_folder, db_path, out_base):
    items = os.listdir(db_path)
    users = [d for d in items if os.path.isdir(os.path.join(db_path, d))]
    tareas = []
    check_path = os.path.join(out_base, input_folder)
    if os.path.exists(check_path) and any(f.endswith('.png') for f in os.listdir(check_path)):
         tareas.append((check_path, os.path.join(out_base, output_folder)))
    else:
        for user in users:
            src = os.path.join(out_base, user, input_folder)
            if os.path.isdir(src):
                tareas.append((src, os.path.join(out_base, user, output_folder)))     
    for src_dir, dst_dir in tareas:
        os.makedirs(dst_dir, exist_ok=True)
        for filename in [f for f in os.listdir(src_dir) if f.lower().endswith('.png')]:
            fase_func(src_dir, dst_dir, filename)

def fase_ecualizar(src, dst, name):
    img = cv.imread(os.path.join(src, name))
    if img is not None:
        cv.imwrite(os.path.join(dst, name), cv.equalizeHist(cv.cvtColor(img, cv.COLOR_BGR2GRAY)))

def fase_bilateral(src, dst, name):
    img = cv.imread(os.path.join(src, name))
    if img is not None:
        cv.imwrite(os.path.join(dst, name), cv.bilateralFilter(cv.cvtColor(img, cv.COLOR_BGR2GRAY), 10, 10, 10))

def fase_sobel(src, dst, name):
    img = cv.imread(os.path.join(src, name))
    if img is not None:
        g = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
        sx = cv.Sobel(g, cv.CV_64F, 1, 0, ksize=3)
        sy = cv.Sobel(g, cv.CV_64F, 0, 1, ksize=3)
        m = cv.magnitude(sx, sy)
        cv.imwrite(os.path.join(dst, name), cv.normalize(m, None, 0, 255, cv.NORM_MINMAX).astype(np.uint8))

def fase_refinar(src, dst, name):
    img = cv.imread(os.path.join(src, name), cv.IMREAD_GRAYSCALE)
    if img is not None:
        h, w = img.shape
        m = int(min(h, w) * 0.03)
        crop = img[m:h-m, m:w-m]
        cl = cv.morphologyEx(crop, cv.MORPH_OPEN, cv.getStructuringElement(cv.MORPH_ELLIPSE, (3, 3)), iterations=1)
        cv.imwrite(os.path.join(dst, name), cv.normalize(cv.medianBlur(cl, 3), None, 0, 255, cv.NORM_MINMAX))

def ejecutar_pipeline_completo(raw_path, out_path):
    print(f"Procesando: {raw_path} -> {out_path}")
    recortar_roi(raw_path, out_path)
    procesar_fase(fase_ecualizar, "roi", "equalized", raw_path, out_path)
    procesar_fase(fase_bilateral, "equalized", "bilateral_filter", raw_path, out_path)
    procesar_fase(fase_sobel, "bilateral_filter", "sobel", raw_path, out_path)
    procesar_fase(fase_refinar, "sobel", "refinadas", raw_path, out_path)


In [2]:
# --- CONFIGURACIÓN ---
DB_PATH = "data"
OUT_PATH = "output"
TEST_PATH = "test"
OUT_TEST_PATH = "output/test"

os.makedirs(OUT_PATH, exist_ok=True)
os.makedirs(TEST_PATH, exist_ok=True)
os.makedirs(OUT_TEST_PATH, exist_ok=True)

# 1. PREPROCESAMIENTO
print("\n--- FASE 1: PREPROCESAMIENTO ---")
ejecutar_pipeline_completo(DB_PATH, OUT_PATH)
ejecutar_pipeline_completo(TEST_PATH, OUT_TEST_PATH)


--- FASE 1: PREPROCESAMIENTO ---
Procesando: data -> output
Procesando: test -> output/test
